# MLflow RAGAS Scores Query

This notebook fetches `ragas_scores.json` artifacts from MLflow runs and computes mean scores grouped by:
- `question_class`
- `subdomain`
- (`question_class`, `subdomain`)

In [5]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [6]:
# Config
MLFLOW_TRACKING_URI = "http://localhost:8567"
EXPERIMENT_NAME =  "langchain-rag-20260416" #None  # Set to a name string to filter one experiment
MAX_RUNS = 200

# Use one of: "latest", "all", or a specific run_id
RUN_SELECTION = "9a60043ae8e94428806a32f7d5c1a17d" #"latest"

RAGAS_TABLE_ARTIFACT = "ragas_scores.json"
METRIC_COLS = [
    # "faithfulness",
    # "context_precision",
    # "context_recall",
    # "answer_relevance",
    # "factual_correctness",
    "factual_correctness_recall"
]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: http://localhost:8567


In [7]:
def list_artifacts_recursive(run_id: str, path: str = ""):
    items = []
    for art in client.list_artifacts(run_id, path):
        items.append(art.path)
        if art.is_dir:
            items.extend(list_artifacts_recursive(run_id, art.path))
    return items


def normalize_logged_table(json_obj) -> pd.DataFrame:
    # mlflow.log_table can be stored in different JSON shapes depending on version.
    if isinstance(json_obj, list):
        return pd.DataFrame(json_obj)

    if isinstance(json_obj, dict):
        if "data" in json_obj and "columns" in json_obj:
            return pd.DataFrame(json_obj["data"], columns=json_obj["columns"])
        return pd.DataFrame(json_obj)

    raise ValueError("Unsupported ragas_scores.json structure")


def load_ragas_table_for_run(run_id: str):
    artifact_paths = list_artifacts_recursive(run_id)

    # Find ragas_scores.json at any artifact depth.
    matches = [p for p in artifact_paths if p.endswith(RAGAS_TABLE_ARTIFACT)]
    if not matches:
        return None, None

    artifact_path = matches[0]
    local_path = client.download_artifacts(run_id, artifact_path)

    with open(local_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    df = normalize_logged_table(payload)
    return df, artifact_path

In [8]:
# Discover runs
if EXPERIMENT_NAME:
    exp = client.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        raise ValueError(f"Experiment not found: {EXPERIMENT_NAME}")
    experiment_ids = [exp.experiment_id]
else:
    experiment_ids = [e.experiment_id for e in client.search_experiments()]

runs_df = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["attribute.start_time DESC"],
    max_results=MAX_RUNS,
)

if runs_df.empty:
    raise ValueError("No MLflow runs found for the current filter.")

print(f"Found {len(runs_df)} runs")
runs_df[["run_id", "experiment_id", "start_time", "status"]].head(10)

Found 22 runs


,run_id,experiment_id,start_time,status
0,e71d9623694a48c1bfd1da6f8a0c1846,18,2026-04-19 15:32:28.009000+00:00,FINISHED
1,1c428e443610401a95dd4301a6da5451,18,2026-04-19 15:27:54.492000+00:00,FINISHED
2,582660284cd846a4ac205928d2a47f97,18,2026-04-19 15:16:18.629000+00:00,FINISHED
3,28e3e838a05f4227b5cd3b696858a72f,18,2026-04-19 14:58:51.341000+00:00,FINISHED
4,9a60043ae8e94428806a32f7d5c1a17d,18,2026-04-19 14:16:34.084000+00:00,FINISHED
5,3758cf2b1ea54fcaabecd084a35238a4,18,2026-04-19 14:03:54.941000+00:00,FINISHED
6,6d3a3b8d27074fa7b84b581c5affe480,18,2026-04-19 12:15:30.045000+00:00,FINISHED
7,9bd16f995dc044d1b9d8c27daa72b971,18,2026-04-18 22:17:19.301000+00:00,FINISHED
8,78ef41b4c4b14911ad16944ac3534d62,18,2026-04-18 20:42:13.082000+00:00,FINISHED
9,d01da6ac6fa34436b5d1962f1da8f367,18,2026-04-18 16:24:09.747000+00:00,FINISHED


In [9]:
# Pull ragas_scores.json from each run
tables = []
misses = []

for run_id in runs_df["run_id"].tolist():
    df, artifact_path = load_ragas_table_for_run(run_id)
    if df is None:
        misses.append(run_id)
        continue

    df = df.copy()
    df["run_id"] = run_id
    df["artifact_path"] = artifact_path
    tables.append(df)

if not tables:
    raise ValueError("No ragas_scores.json artifacts found in discovered runs.")

all_scores = pd.concat(tables, ignore_index=True)

for col in METRIC_COLS:
    if col in all_scores.columns:
        all_scores[col] = pd.to_numeric(all_scores[col], errors="coerce")

print(f"Runs with ragas_scores.json: {all_scores['run_id'].nunique()}")
print(f"Runs without ragas_scores.json: {len(misses)}")
all_scores.head()

Runs with ragas_scores.json: 22
Runs without ragas_scores.json: 0


/tmp/ipykernel_18477/3114235660.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_scores = pd.concat(tables, ignore_index=True)


,user_input,response,reference,retrieved_contexts,retrieved_context,subdomain,question_class,context_recall,factual_correctness_recall,metric_errors,run_id,artifact_path,factual_correctness,context_precision
0,Does the QProv definition of circuit width inc...,"No, the QProv definition of circuit width refe...",The QProv specification defines circuit width ...,[---\n\n### Q4 — Circuit Width\n\n**Field ID:*...,---\n\n### Q4 — Circuit Width\n\n**Field ID:**...,qprov_provenance_taxonomy,fact_single,1.0,1.0,{},e71d9623694a48c1bfd1da6f8a0c1846,ragas_scores.json,NaN,NaN
1,How many qubits are typically found in a NISQ-...,NISQ (Noisy Intermediate-Scale Quantum) techno...,Quantum computers in the NISQ era typically co...,[Häner et al. [26] introduced a methodology to...,Häner et al. [26] introduced a methodology to ...,nisq_constraints_and_qsd_constraints,fact_single,1.0,1.0,{},1c428e443610401a95dd4301a6da5451,ragas_scores.json,NaN,NaN
2,How many qubits are typically found in a NISQ-...,NISQ-era (Noisy Intermediate-Scale Quantum) qu...,Quantum computers in the NISQ era typically co...,[## **3.2** | **Quantum computer category** \...,## **3.2** | **Quantum computer category** \n...,nisq_constraints_and_qsd_constraints,fact_single,1.0,1.0,{},582660284cd846a4ac205928d2a47f97,ragas_scores.json,NaN,NaN
3,How many qubits are typically found in a NISQ-...,"Typically, NISQ (Noisy Intermediate-Scale Quan...",Quantum computers in the NISQ era typically co...,[],,nisq_constraints_and_qsd_constraints,fact_single,NaN,1.0,{'context_recall': 'ValueError: retrieved_cont...,28e3e838a05f4227b5cd3b696858a72f,ragas_scores.json,NaN,NaN
4,How many qubits are typically found in a NISQ-...,NISQ-era quantum computers typically have betw...,Quantum computers in the NISQ era typically co...,[],,nisq_constraints_and_qsd_constraints,fact_single,NaN,1.0,{'context_recall': 'ValueError: retrieved_cont...,9a60043ae8e94428806a32f7d5c1a17d,ragas_scores.json,NaN,NaN


In [10]:
# Select target rows for aggregation
if RUN_SELECTION == "latest":
    latest_run_id = runs_df.iloc[0]["run_id"]
    target = all_scores[all_scores["run_id"] == latest_run_id].copy()
    print(f"Using latest run: {latest_run_id}")
elif RUN_SELECTION == "all":
    target = all_scores.copy()
    print("Using all runs with ragas_scores.json")
else:
    target = all_scores[all_scores["run_id"] == RUN_SELECTION].copy()
    if target.empty:
        raise ValueError(f"No rows found for run_id={RUN_SELECTION}")
    print(f"Using selected run: {RUN_SELECTION}")

required_cols = ["question_class", "subdomain"]
for c in required_cols:
    if c not in target.columns:
        raise ValueError(f"Missing required column in ragas table: {c}")

target[["run_id", "user_input", "question_class", "subdomain"] + [c for c in METRIC_COLS if c in target.columns]].head()

Using selected run: 9a60043ae8e94428806a32f7d5c1a17d


,run_id,user_input,question_class,subdomain,factual_correctness_recall
4,9a60043ae8e94428806a32f7d5c1a17d,How many qubits are typically found in a NISQ-...,fact_single,nisq_constraints_and_qsd_constraints,1.00
5,9a60043ae8e94428806a32f7d5c1a17d,What are the primary challenges quantum resear...,summary,nisq_constraints_and_qsd_constraints,0.69
6,9a60043ae8e94428806a32f7d5c1a17d,Why does the accuracy of my results typically ...,reasoning,nisq_constraints_and_qsd_constraints,0.89
7,9a60043ae8e94428806a32f7d5c1a17d,What is the specific hourly cost for accessing...,unanswerable,nisq_constraints_and_qsd_constraints,1.00
8,9a60043ae8e94428806a32f7d5c1a17d,What is the fundamental unit of organization f...,fact_single,experiment_tracking_fundamentals,1.00


In [11]:
metric_cols_present = [c for c in METRIC_COLS if c in target.columns]

avg_by_question_class = (
    target.groupby("question_class", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_subdomain = (
    target.groupby("subdomain", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_qclass_and_subdomain = (
    target.groupby(["question_class", "subdomain"], dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

print("Average by question_class")
display(avg_by_question_class)

print("Average by subdomain")
display(avg_by_subdomain)

print("Average by (question_class, subdomain)")
display(avg_by_qclass_and_subdomain)

Average by question_class


,factual_correctness_recall
question_class,
fact_single,0.960
reasoning,0.716
summary,0.782
unanswerable,0.700


Average by subdomain


,factual_correctness_recall
subdomain,
experiment_tracking_fundamentals,0.7375
mlflow_tracking_api,0.5875
nisq_constraints_and_qsd_constraints,0.8950
qiskit-specific_experiment_tracking_using_mlflow_and_qprov,0.8875
qprov_provenance_taxonomy,0.8400


Average by (question_class, subdomain)


factual_correctness_recall
question_class subdomain                                                                     
fact_single    experiment_tracking_fundamentals                                          1.00
               mlflow_tracking_api                                                       1.00
               nisq_constraints_and_qsd_constraints                                      1.00
               qiskit-specific_experiment_tracking_using_mlflo...                        0.80
               qprov_provenance_taxonomy                                                 1.00
reasoning      experiment_tracking_fundamentals                                          0.67
               mlflow_tracking_api                                                       0.60
               nisq_constraints_and_qsd_constraints                                      0.89
               qiskit-specific_experiment_tracking_using_mlflo...                        0.75
               qprov_provenance_taxonomy                                                 0.67
summary        experiment_tracking_fundamentals                                          0.78
               mlflow_tracking_api                                                       0.75
               nisq_constraints_and_qsd_constraints                                      0.69
               qiskit-specific_experiment_tracking_using_mlflo...                        1.00
               qprov_provenance_taxonomy                                                 0.69
unanswerable   experiment_tracking_fundamentals                                          0.50
               mlflow_tracking_api                                                       0.00
               nisq_constraints_and_qsd_constraints                                      1.00
               qiskit-specific_experiment_tracking_using_mlflo...                        1.00
               qprov_provenance_taxonomy                                                 1.00